DSC550-T302<br>
Week 3 Assignment<br>
Jeremy Barton

# Preparing Data for a Custom Model, "Bag of Words" Method

## Part 1: Using the TextBlob Sentiment Analyzer

#### Introduction

This notebook will classify movies review as positive or neative assuming a polarity score of >= 0 as positive, and <= 0 as negative 

The TextBlob model will then be compared to random guessing to prove the accuracy benefits of using statistics model. 

Additionaly, the RoBERTa model will be applied to the reviews and checked using the same accuracy method.

### 1) Import the movie review data as a data frame and ensure that the data is loaded properly.

Ensure there is a `sep='\t'` after the path to the data so pandas knows it's a tab delimited file. Otherwise an error will return.

In [26]:
import pandas as pd
from textblob import TextBlob
from sklearn.metrics import accuracy_score
from transformers import pipeline
import torch as torch
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
nltk.download('punkt')


# using sep='\t' tells pandas this is a 
#   tab separated document
movie_reviews = pd.read_csv("labeledTrainData.tsv", sep='\t')

[nltk_data] Downloading package stopwords to /home/jeremy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/jeremy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Check the data exists.

In [27]:
movie_reviews.head(5)

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


In [28]:
movie_reviews.shape

(25000, 3)

### 2) How many of each positive and negative reviews are there?

To calculate this the 1s and 0s will be totalled up using `.value_counts()`. The below line sums the 1s and 0s of the sentiment column from the movie_reviews dataset.

<b>Positive</b>

In [29]:
# .int() turns the result into an integeer
# .sum() is used on the sentiment indicator "1"
#   from the "movie_reviews" dataframe
positive = int((movie_reviews['sentiment'] == 1).sum())
positive

12500

<b>Negative</b>

In [30]:
negative = int((movie_reviews['sentiment'] == 0).sum())
negative

12500

### 3) Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.

In [31]:
# create analysis container
analysis_df = []

# iterate through movie reviews
#   and collect text 
for text in movie_reviews['review']:
    # calculate polarity score using TextBlob
    pol = TextBlob(text).sentiment.polarity
    # use TextBlob's build in .sentiment.polarity
    #   core usage methods

    # checking if the polarity is greater than
    # or equal to zero
    if pol >=0:
        sentiment = 1 # marks a positive review
    else:
        sentiment = 0 # marks a negative review
    
    # append the results to a list
    analysis_df.append({
        'review': text,
        'polarity': pol,
        'predicted': sentiment
    })

After that's finished running, the results can now be printed.

In [32]:
# convert to dataframe and add colums
#   to compare actual and predicted results
analysis_df = pd.DataFrame(analysis_df)
analysis_df['actual'] = movie_reviews['sentiment'].values

print(analysis_df.head())

                                              review  polarity  predicted  \
0  With all this stuff going down at the moment w...  0.001277          1   
1  \The Classic War of the Worlds\" by Timothy Hi...  0.256349          1   
2  The film starts with a manager (Nicholas Bell)... -0.053941          0   
3  It must be assumed that those who praised this...  0.134753          1   
4  Superbly trashy and wondrously unpretentious 8... -0.024842          0   

   actual  
0       1  
1       1  
2       0  
3       0  
4       1  


In the dataframe, the "predicted" represents what TextBlob calculated the sentiment was. The "actual" column is what the actual sentiment was according to the data.

### 4) Check the accuracy of this model. Is this model better than random guessing?

Checking the accuracy of the results can be performed using <b>scikit-learn's</b> `accuracy_score`. It compares two columns by taking the true and predicted labels, counts how many `predicted` values match the `actual` values and divides the total number of rows.

In [33]:
# accuracy_score() takes two parameters
#   the actual results and the predicted values
accuracy = accuracy_score(analysis_df['actual'], analysis_df['predicted'])
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 68.52%


So with a total of 25,000 total reviews, a 68.52% accuracy this means 17,130 reviews were correctly classified, and 7,870 were incorrectly classified.

<b>What are some issues with this test?</b>

TextBlob is a general purpose sentiment tool, which means it isn't traied for movie reviews specifically. As typically plauged with in natural language processing these movie reviews may include, sarcasm, nuance and complex that simpler polarity scoring techniques might miss.

<b>Is this better than random guessing?</b>

68.52% is more accurate than 50%, so this would statistically prove that using TextBlob is a better sentiment analyzer than than taking a random guess.

<b>Additional sentiment analyzers</b>

### 5. For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

#### RoBERTa Model

Another model that can be used to analyze sentiment is RoBERTa (Robustly Optimized BERT Pretraining Approach). It understands contextual sentiments <b>how</b> entities are discussed. It is developed by Faceook AI (not Meta AI), pretrained on social sentiment data on Twitter, built on Google's BERT model but with improvements for language understanding tasks like sentiment classification. and returns confidence scored for positive, neurtal and negative entities.

Define the classifier to be used by torch.

In [34]:
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", # get RoBERTa model pipeline from PyTorch
    truncation=True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 26535.37it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RoBERTa will predict a sentiment label for each review (positive, negative, neutral) and attach a confidence score. This part of the process is visualized in the below output. It is then converted to a 1 or 0 to macth the movie_reviews format.

In [35]:
roberta_results = []

for text in movie_reviews['review']:
    result = classifier(text[:512])[0]  # truncate to model's max length
    # LABEL_0 = negative, LABEL_2 = positive
    if result['label'] in ['LABEL_2', 'positive']:
        # label positive and negative predictions
        #  where 1 = pos, 0 = neg
        sentiment = 1
    else:
        sentiment = 0
    # add newly predicted results to the roberta_results list
    roberta_results.append(sentiment)
    
# store predictions as a new column in the dataframe
movie_reviews['roberta_predicted'] = roberta_results

Explore the added `roberta_predicted` column to see the results.

In [36]:
movie_reviews

,id,sentiment,review,roberta_predicted
0,5814_8,1,With all this stuff going down at the moment w...,0
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi...",1
2,7759_3,0,The film starts with a manager (Nicholas Bell)...,0
3,3630_4,0,It must be assumed that those who praised this...,0
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...,1
...,...,...,...,...
24995,3453_3,0,It seems like more consideration has gone into...,0
24996,5064_1,0,I don't believe they made this film. Completel...,0
24997,10905_3,0,"Guy is a loser. Can't get girls, needs to buil...",0
24998,10194_3,0,This 30 minute documentary Buñuel made in the ...,0


Using `scikit-learn`'s accuracy_score will help to determine the accuracy of this model. 

The formula is:

`accuracy = # correct predictions / total predictions`

In [37]:
roberta_accuracy = accuracy_score(movie_reviews['sentiment'], movie_reviews['roberta_predicted'])
print(f"RoBERTa Accuracy: {roberta_accuracy:.2%}")

RoBERTa Accuracy: 75.80%


One of the downsides of using the RoBERTa model is speed. Running 25,000 reviews on a budget laptop could take up to an hour. This exercise was performed on an NVIDIA GeForce GTX 1660 SUPER GPU.

## Part 2: Prepping Text for a Custom Model

If you want to run your own model to classify text, it needs to be in proper form to do so. The following steps will outline a procedure to do this on the movie reviews text.

### 1. Convert all text to lowercase letters.

In [38]:
# for each item in the range
#   of movie reviews
for i in range(len(movie_reviews)):
    movie_reviews.loc[i, 'review'] = movie_reviews['review'][i].lower()
    # choose the review column and apply .lower() to it

In [39]:
# Check changes
movie_reviews['review'].head(3)

0    with all this stuff going down at the moment w...
1    \the classic war of the worlds\" by timothy hi...
2    the film starts with a manager (nicholas bell)...
Name: review, dtype: str

In [40]:
# pandas equivilant
movie_reviews['review'] = movie_reviews['review'].str.lower()

### 2. Remove punctuation and special characters from the text.

In the code, `r'[^a-zA-Z0-9\s]'`,

`r'...'` — the r makes it a raw string so Python doesn't interpret backslashes as escape characters<br>
`[ ]` — defines a set of characters to match<br>
`^` — when inside brackets, this means "NOT" — so it flips the match to everything outside the set<br>
`a-zA-Z` — all lowercase and uppercase letters<br>
`0-9` — all digits<br>
`\s` — any whitespace (spaces, tabs, newlines)

In [41]:
movie_reviews['review'] = movie_reviews['review'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

In [42]:
# check results
movie_reviews['review'].str.contains(r'[^a-zA-Z0-9\s]', regex=True).sum()  
# should return 0

np.int64(0)

### 3. Remove stop words.

NLTK's stopwords function should work well to remove the stopwords which have no true meaning in the analysis.

In [43]:
stop_words = set(stopwords.words('english'))

# get all tokens from all reviews
all_tokens = movie_reviews['review'].str.lower().apply(word_tokenize)

# filter out stopwords
filtered_tokens = all_tokens.apply(
    lambda x: [word for word in x if word.isalpha() and word not in stop_words]
)

The below output shows the reviews column with stopwords removed.

In [44]:
movie_reviews['review'].head()

0    with all this stuff going down at the moment w...
1    the classic war of the worlds by timothy hines...
2    the film starts with a manager nicholas bell g...
3    it must be assumed that those who praised this...
4    superbly trashy and wondrously unpretentious 8...
Name: review, dtype: str

### 4. Apply NLTK’s PorterStemmer.

According to the defenition in <i>Machine Learning with Python Bookbook: Practival Solutions from Preprocessing to Deep Learning (2023, O'Reilly)</i>, Stemming reduces a word to its stem by identifying and removing "affixes" while keeping the root meaning of the word.

An example they give is the word "tradition" and "traditional". They both have the pattern "tradit" in its word which is called the stem. Although they are two different words, they both represent the same general concept.

In [45]:
# make porter stemmer object
porter = PorterStemmer()

# apply stemming to filtered tokens (already has stopwords removed)
stemmed_tokens = filtered_tokens.apply(
    lambda tokens: [porter.stem(word) for word in tokens]
)

# display results
stemmed_tokens.head()

0    [stuff, go, moment, mj, ive, start, listen, mu...
1    [classic, war, world, timothi, hine, entertain...
2    [film, start, manag, nichola, bell, give, welc...
3    [must, assum, prais, film, greatest, film, ope...
4    [superbl, trashi, wondrous, unpretenti, exploi...
Name: review, dtype: object

### 5. Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.

Lists of token must be converted back to strings be read by the CountVectorizer().

In [46]:
# convert lists of tokens back to strings 
# by joining them with spaces
stemmed_text = stemmed_tokens.apply(lambda tokens: ' '.join(tokens))
                                    # lambda function used to join space with a space


Convert to bag-of-words using `CountVectorizer()`

In [47]:
# create bag-of-words matrix
count = CountVectorizer()
# count = DictVectorizer()
bow = count.fit_transform(stemmed_text)
                    # fit_transform used to scale and standardize
                    #   the data for fit the bag-of-words 

# show dimensions
print(f"Bag-of-words matrix shape: {bow.shape}") # bag of words shape
print(f"Number of reviews (rows): {bow.shape[0]}")# number of reviews
print(f"Number of unique words (columns): {bow.shape[1]}") # unique words

Bag-of-words matrix shape: (25000, 89118)
Number of reviews (rows): 25000
Number of unique words (columns): 89118


### 6. Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.

In [48]:
# create the if-idf feature matrix
tfidf = TfidfVectorizer()
feature_matrix = tfidf.fit_transform(stemmed_text)

Checking that the shape of the matrix matches the bag-of-words.

In [49]:
feature_matrix.shape

(25000, 89118)

This shape looks identical to the bag-of-words matrix.

Check the contents of the bag-of-words.

In [ ]:
# print(feature_matrix)

(25000, 89118)